<a href="https://colab.research.google.com/github/bberrium/Picsart-Academy/blob/main/Advanced_CV_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: Advanced Computer Vision

Welcome to your final Computer Vision assignment! Today, you will prove your mastery over the two dominant branches of modern CV: **Semantic Segmentation** and **Object Detection**.

**Your Mission:**
1. **Part 1: Build U-Net from Scratch.** You will implement the famous U-Net architecture for Semantic Segmentation.
2. **Part 2: YOLO Fine-Tuning & Video Inference.** You will use a state-of-the-art YOLO model, fine-tune it, and run it on a real-world video.

## Part 1: U-Net (The "Tensor Tetris" Challenge)

**EXPLICIT WARNING REGARDING AI**

Do **NOT** use AI to write this U-Net code.

Why? Because an AI will output the correct code in 0.5 seconds, and you will learn absolutely nothing. The entire point of this exercise is to struggle with "Tensor Tetris"—the art of keeping track of spatial dimensions, channel counts, and concatenations in your head.

If you don't fight with the shape mismatches here, you will be completely lost when you have to debug a real neural network on the job. Use documentation, use your notes, but write the code yourself.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### 1.1 The Double Convolution Block
In U-Net, every step of the encoder and decoder features two consecutive 3x3 Convolutions, each followed by a ReLU activation. (Modern implementations also use BatchNorm).

**Task:** Implement the `DoubleConv` block.
* `Conv2d` -> `BatchNorm2d` -> `ReLU` -> `Conv2d` -> `BatchNorm2d` -> `ReLU`
* Make sure `padding=1` in your Conv2d layers so the spatial dimensions don't shrink!

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # TODO: Define the sequential block described above
        self.double_conv = nn.Sequential(
            # ... your code here ...
        )

    def forward(self, x):
        return self.double_conv(x)

### 1.2 The U-Net Architecture
Now, assemble the U-Net.

**The Encoder (Down):**
* Uses your `DoubleConv`.
* Uses `nn.MaxPool2d(2)` to halve the spatial dimensions.

**The Decoder (Up):**
* Uses `nn.ConvTranspose2d` to double the spatial dimensions.
* Uses `torch.cat` to concatenate the skip connection from the encoder.
* Uses your `DoubleConv` to process the concatenated feature map.


In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # --- ENCODER (Downsampling) ---
        # Image goes from 3 channels -> 64 channels
        self.inc = DoubleConv(in_channels, 64)

        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        # TODO: Define down2 (128 -> 256) and down3 (256 -> 512)
        # self.down2 = ...
        # self.down3 = ...

        # The Bottleneck (512 -> 1024)
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # --- DECODER (Upsampling) ---
        # TODO: Define the upsampling layers.
        # HINT: The ConvTranspose2d cuts the channels in half (e.g. 1024 -> 512).
        # Then, you concatenate a skip connection of 512 channels, bringing the total back to 1024.
        # Finally, the DoubleConv processes that 1024 back down to 512.

        self.up1_transpose = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up1_conv = DoubleConv(1024, 512)

        # self.up2_transpose = ...
        # self.up2_conv = ...

        # self.up3_transpose = ...
        # self.up3_conv = ...

        # self.up4_transpose = ...
        # self.up4_conv = ...

        # --- FINAL OUTPUT ---
        # A simple 1x1 convolution to map the channels to the number of classes
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # ENCODER
        x1 = self.inc(x)
        x2 = self.down1(x1)
        # TODO: Pass through down2, down3, down4
        # x3 = ...
        # x4 = ...
        # x5 = ... # This is the bottleneck

        # DECODER
        # Step 1: Upsample the bottleneck
        # x_up = self.up1_transpose(x5)

        # Step 2: Concatenate with the corresponding skip connection (x4)
        # HINT: torch.cat([x_up, x4], dim=1)
        # x_concat = ...

        # Step 3: Pass through the double convolution
        # x_out = self.up1_conv(x_concat)

        # TODO: Repeat for up2, up3, and up4 using the skip connections x3, x2, and x1

        # TODO: Pass the final feature map through self.outc
        # logits = ...

        # return logits
        pass # Remove this when you implement the function


### 1.3 The U-Net Sanity Check
If you wired your network correctly, this cell will run without error and output a tensor of shape `[1, 2, 256, 256]`.
If it crashes, read the error carefully. It will tell you exactly which layers have mismatched channel counts or spatial dimensions!


In [ ]:
print("Testing U-Net Architecture...")
model = UNet(in_channels=3, num_classes=2)
dummy_image = torch.randn(1, 3, 256, 256)

try:
    output = model(dummy_image)
    print(f"Success! Output shape: {output.shape}")
    assert output.shape == (1, 2, 256, 256), "Output shape is incorrect!"
except Exception as e:
    print(f"FAILED! Error: {e}")


## Part 2: YOLO Fine-Tuning & Video Inference

Writing architectures from scratch is for academia. In the real world, you use `Ultralytics`.
In this section, you will install the Ultralytics library, fine-tune a YOLOv8 model on a toy dataset, and then use it to track objects in a video.

In [ ]:
# NOTE: If you are running this locally, you may need to run `!pip install ultralytics` in a separate cell first.
import urllib.request
import os

try:
    from ultralytics import YOLO
    print("Ultralytics YOLO is installed and ready!")
except ImportError:
    print("Please install ultralytics: !pip install ultralytics")

# Download a sample traffic video for inference
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
video_path = "traffic_video.mp4"

if not os.path.exists(video_path):
    print("Downloading sample video...")
    urllib.request.urlretrieve(video_url, video_path)
    print("Download complete.")


Please install ultralytics: pip install ultralytics
Download complete.


### 2.1 Fine-Tune YOLOv8

**Task:** Load a pre-trained `yolov8n.pt` (Nano) model. Train it for exactly **3 epochs** on the `coco8.yaml` dataset (a tiny, built-in dataset containing 8 images just for testing code pipelines).

*Hint: This requires exactly two lines of code using the Ultralytics API.*

In [ ]:
# TODO: 1. Initialize the YOLO model with 'yolov8n.pt'
# model_yolo = ...

# TODO: 2. Call the train() method on your model.
# Arguments to pass: data='coco8.yaml', epochs=3, imgsz=640
# ...

### 2.2 Inference on a Video

Object detection models don't just process static images. They process videos frame-by-frame.

**Task:** Use your fine-tuned model to run predictions on `traffic_video.mp4`.
You want to **save** the output so you can watch the video with the bounding boxes drawn on it!

*Hint:* Use the `predict()` method on your model. Look up the documentation on how to pass `source` and `save=True`.


In [ ]:
print(f"Running inference on {video_path}...")

# TODO: Run the model's predict method on the video_path.
# Make sure to tell it to save the output!
# ...

print("Inference complete! Check the 'runs/detect/predict' folder in your directory to watch your annotated video.")